# Phương pháp CPU: TF-IDF theo ngữ cảnh từng khía cạnh

Máy không có GPU: dùng PyVi + NB-SVM theo mệnh đề, kết hợp thông tin toàn văn.
Nhãn do mô hình học từ train dự đoán. Từ khóa chỉ dùng để chọn vùng đặc trưng.
Không dùng LLM/API để sinh nhãn test. Không có dữ liệu thêm.

Mục tiêu đề nghị 0,98 **chưa được chứng minh**. Dùng ABSA-F1 macro của 14 nhãn
khen/chê để so sánh; không thay bằng Accuracy. Validation đã được tham khảo ở
những vòng phát triển trước; không phải holdout xác nhận hoàn toàn mới.

Chạy tại thư mục absa_solution; dữ liệu giảng viên nằm ở thư mục cha. Cài
`python -m pip install -r requirements_cpu.txt`. Nếu đổi vị trí, đặt ABSA_DATA_DIR.
Notebook dùng kết quả cục bộ đã huấn luyện nếu có; đặt FORCE_TRAIN=True để fit
lại từ đầu. Khi tải code từ GitHub chưa có kết quả, notebook tự huấn luyện.

In [1]:
import os, json, sys
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
from threadpoolctl import threadpool_limits
import scoped_cpu
import score_absa
DATA=Path(os.environ.get('ABSA_DATA_DIR','..')).resolve()
OUT=Path('scoped_results')
FORCE_TRAIN=False
print('Python',sys.version.split()[0],'CPU; không dùng GPU')

Python 3.12.14 CPU; không dùng GPU


## Huấn luyện hoặc nạp thực nghiệm đã chạy

2.812 fit / 938 tuning / 938 validation, nhóm theo văn bản chuẩn hóa, seed 2026.
Chọn C, ngưỡng, và trọng số local/global trên tuning. Chỉ sau đó tính validation.
Vectorizer fit riêng trên fit trong đánh giá. Cuối cùng fit lại trên toàn train
và tạo submission; vẫn giữ ngưỡng đã chọn. Không dùng test để fit vectorizer.

In [2]:
if FORCE_TRAIN or not (OUT/'metrics.json').exists():
    with threadpool_limits(limits=1):
        report=scoped_cpu.fit(DATA,OUT)
else:
    print('Dùng thực nghiệm đã huấn luyện cục bộ. FORCE_TRAIN=True để chạy lại.')
    report=json.loads((OUT/'metrics.json').read_text())
score_absa.show(report)
info=json.loads((OUT/'run_info.json').read_text())
print(json.dumps(info,ensure_ascii=False,indent=2))

Dùng thực nghiệm đã huấn luyện cục bộ. FORCE_TRAIN=True để chạy lại.
Số mẫu: 938
ABSA-F1 (điểm xếp hạng) = 0.782   CI95% [0.765, 0.797]
F1 theo cực: khen 0.864 | chê 0.699 | trung tính (phụ) 0.010 | phát hiện khía cạnh (phụ) 0.899
Nhãn yếu nhất: gia_tri_che F1 = 0.450; độ lệch chuẩn F1 14 nhãn = 0.140
Precision TB = 0.809 | Recall TB = 0.768 | Accuracy TB = 0.935 (14 nhãn)
  khen: P 0.875 R 0.854 | chê: P 0.743 R 0.682
kappa TB = 0.738 | MCC TB = 0.742 | Balanced acc TB = 0.863
F1 vi mô: khía cạnh = 0.909 | (khía cạnh, cực) = 0.808 | khớp đúng cả đánh giá = 0.416
Mốc lớp đa số =   nan; cải thiện tương đối =   nan

Khía cạnh      Nhắc F1 | Khen F1     P     R     κ     n | Chê F1      P     R     κ     n | Trung tính F1  n
nhan_vien      0.942  | 0.936 0.941 0.931 0.865   495 | 0.704  0.635 0.791 0.660   110 | 0.000       0
sach_se        0.967  | 0.976 0.971 0.981 0.964   310 | 0.825  0.881 0.776 0.811    76 | 0.000       0
vi_tri         0.917  | 0.901 0.907 0.895 0.851   315 | 0.606 

## So sánh và chỉ số

In [3]:
# Các baseline dưới đây là số đo thực tế của những lần chạy trước trên cùng split.
# Không phải kết quả chạy lại baseline trong cell này.
comparison=pd.DataFrame([
 {'method':'NB-SVM ban đầu','validation_ABSA_F1':0.7561255284105146},
 {'method':'Tách từ + ensemble','validation_ABSA_F1':0.7655857952322037},
 {'method':'Ngữ cảnh theo khía cạnh','validation_ABSA_F1':report['ABSA_F1']}])
display(comparison)
print('Bootstrap CI95%:',report['ABSA_F1_CI95'])
print({k:report[k] for k in ['precision_TB','recall_TB','accuracy_TB','kappa_TB','mcc_TB','bal_acc_TB']})
rows=[]
for aspect,d in report['theo_khia_canh'].items():
    for polarity in ['khen','che']:
        rows.append({'aspect':aspect,'polarity':polarity,**d[polarity]})
display(pd.DataFrame(rows))
print('Nhãn khó nhất:',report['F1_nhan_thap_nhat'])
print('Khía cạnh khó nhất:',min(report['theo_khia_canh'],key=lambda a:sum(report['theo_khia_canh'][a][p]['f1'] for p in ['khen','che'])/2))

,method,validation_ABSA_F1
0,NB-SVM ban đầu,0.756126
1,Tách từ + ensemble,0.765586
2,Ngữ cảnh theo khía cạnh,0.781506


Bootstrap CI95%: [0.7645133498462879, 0.797113718395018]
{'precision_TB': 0.8091655811662176, 'recall_TB': 0.7681035521046354, 'accuracy_TB': 0.9348157173317088, 'kappa_TB': 0.7384800431064267, 'mcc_TB': 0.7422858923717682, 'bal_acc_TB': 0.863466858345279}


,aspect,polarity,f1,precision,recall,accuracy,kappa,mcc,bal_acc,ho_tro
0,nhan_vien,khen,0.936041,0.940816,0.931313,0.932836,0.865337,0.865387,0.932925,495
1,nhan_vien,che,0.704453,0.635036,0.790909,0.922175,0.660256,0.665532,0.865261,110
2,sach_se,khen,0.975923,0.971246,0.980645,0.984009,0.963952,0.963977,0.983157,310
3,sach_se,che,0.825175,0.880597,0.776316,0.973348,0.810811,0.812698,0.883518,76
4,vi_tri,khen,0.900958,0.906752,0.895238,0.933902,0.851362,0.851401,0.924345,315
5,vi_tri,che,0.605505,0.600000,0.611111,0.954158,0.581172,0.581199,0.793112,54
6,phong,khen,0.836478,0.833856,0.839117,0.889126,0.752608,0.752617,0.876885,317
7,phong,che,0.816399,0.797909,0.835766,0.890192,0.738132,0.738536,0.874209,274
8,gia_tri,khen,0.896296,0.916667,0.876812,0.970149,0.878872,0.879168,0.931531,138
9,gia_tri,che,0.450000,0.720000,0.327273,0.953092,0.429077,0.465818,0.659673,55


Nhãn khó nhất: {'nhan': 'gia_tri_che', 'f1': 0.45}
Khía cạnh khó nhất: tien_nghi


## Phân tích năm lỗi từ validation

Các dòng dưới đây là train được giữ lại làm validation. Không có text test.
Bản công khai GitHub bỏ output nguyên văn đánh giá; chạy cục bộ sẽ tạo lại.
Nhận xét tự động chỉ mô tả loại lỗi dựa trên nhãn sai, chưa thay thế phân tích
ngữ nghĩa thủ công của người làm bài.

In [ ]:
truth=pd.read_csv(OUT/'validation_truth.csv').set_index('id')
pred=pd.read_csv(OUT/'validation_prediction.csv').set_index('id').loc[truth.index]
train=pd.read_csv(DATA/'train.csv').set_index('id')
examples=[]
for idx in truth.index:
    wrong=[c for c in score_absa.COLS if truth.loc[idx,c]!=pred.loc[idx,c]]
    if wrong:
        examples.append({'id':str(idx),'text':train.loc[idx,'text'],
                         'errors':{c:{'truth':int(truth.loc[idx,c]),'prediction':int(pred.loc[idx,c])} for c in wrong}})
(OUT/'validation_errors.json').write_text(json.dumps(examples,ensure_ascii=False,indent=2))
for e in examples[:5]:
    print(e['id'],e['text'])
    print(e['errors'])
    missed=[c for c,d in e['errors'].items() if d['truth']==1]
    extra=[c for c,d in e['errors'].items() if d['truth']==0]
    print('Bỏ sót:',missed,'; dự đoán thừa:',extra)
    print('Cần kiểm tra: phạm vi mệnh đề, tín hiệu gián tiếp, phủ định và giao nhau giữa khía cạnh.')
    print()

## Tái lập suy luận và kiểm tra định dạng

In [5]:
with threadpool_limits(limits=1):
    scoped_cpu.predict(OUT/'model.joblib',DATA/'test_blind.csv',OUT/'submission_reproduced.csv')
a=(OUT/'submission.csv').read_bytes()
b=(OUT/'submission_reproduced.csv').read_bytes()
assert a==b
assert score_absa.kiem_tra(OUT/'submission.csv',DATA/'test_blind.csv')==0
print('Nạp model trong suy luận: CSV khớp chính xác.')
print('Không thể suy ra điểm private từ validation; chưa xác nhận đạt 0,98.')

File hợp lệ, có thể nộp.
File hợp lệ, có thể nộp.
Nạp model trong suy luận: CSV khớp chính xác.
Không thể suy ra điểm private từ validation; chưa xác nhận đạt 0,98.


## Giới hạn

Từ khóa có thể bỏ sót cách nói gián tiếp và tách sai mệnh đề; thông tin toàn văn
vẫn cần để bù lại. Tuning nhiều cấu hình có thể quá khớp tuning, đặc biệt nhãn
hiếm. Validation đã được xem qua nhiều vòng phát triển. CI bootstrap là bất định
lấy mẫu có điều kiện với mô hình cố định, không mô tả dịch chuyển train/test.

PhoBERT chỉ mới kiểm tra backward trên CPU, chưa có ABSA-F1 huấn luyện đầy đủ.
Không dùng điểm Accuracy cao để khẳng định đạt mục tiêu ABSA-F1. Không có dữ
liệu thêm hoặc thí nghiệm ablation dữ liệu thêm.